# 10 — Synthetic-to-Real Ratio Ablation

## Objective

This notebook evaluates whether the full 1:1 synthetic-to-real ratio is necessary for robust drone–bird micro-Doppler classification.

Three training configurations are compared:

1. **0:1 — real-only:** 1,150 real observations.
2. **0.5:1 — reduced augmentation:** 1,150 real and 576 synthetic observations.
3. **1:1 — full augmentation:** 1,150 real and 1,150 synthetic observations.

The completed 0:1 and 1:1 results from Notebook 09 are reused. Only the 0.5:1 configuration is newly trained across seeds 42, 52, 62, 72, and 82.

The synthetic observations are transformation-derived children of the 10% real subset, not independently simulated radar acquisitions; consequently, the experiment measures augmentation benefit rather than an increase in the number of independent real recording sessions.

The 0.5:1 synthetic subset is selected once using a fixed selection seed and remains identical across all model-training seeds. Ratio selection is based on validation performance rather than test performance.

## 1. Reproducibility and Experiment Controls

The cells below define the fixed seeds, data locations, protected output paths, and safe training switch. The saved notebook leaves `RUN_RATIO_TRAINING = False`, so restarting the kernel and running the notebook will not retrain or overwrite completed models.

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from tensorflow.keras import (
    layers,
    models,
    regularizers
)

In [ ]:
BATCH_SIZE = 64
MAX_EPOCHS = 50

MODEL_SEEDS = [
    42,
    52,
    62,
    72,
    82
]

SYNTHETIC_SELECTION_SEED = 42

REAL_SAMPLES_PER_CLASS = 575

HALF_RATIO_SYNTHETIC_PER_CLASS = 288

HALF_RATIO_SYNTHETIC_SAMPLES = (
    2
    * HALF_RATIO_SYNTHETIC_PER_CLASS
)

TARGET_SYNTHETIC_RATIO = 0.5

ACTUAL_SYNTHETIC_RATIO = (
    HALF_RATIO_SYNTHETIC_SAMPLES
    / (2 * REAL_SAMPLES_PER_CLASS)
)

RATIO_CONFIGURATION = (
    "real_plus_synthetic_0_5_to_1"
)

# Safe default. Change only after all
# validation and protection checks pass.
RUN_RATIO_TRAINING = False

print("TensorFlow version:", tf.__version__)

print(
    "Available GPUs:",
    tf.config.list_physical_devices(
        "GPU"
    )
)

print("Model seeds:", MODEL_SEEDS)

print(
    "Synthetic selection seed:",
    SYNTHETIC_SELECTION_SEED
)

print(
    "Target synthetic ratio:",
    TARGET_SYNTHETIC_RATIO
)

print(
    "Actual synthetic ratio:",
    round(
        ACTUAL_SYNTHETIC_RATIO,
        6
    )
)

print(
    "Run ratio training:",
    RUN_RATIO_TRAINING
)

In [ ]:
OFFICIAL_DATA_DIR = Path(
    "../data/processed/official_split"
)

LIMITED_DATA_DIR = Path(
    "../data/processed/limited_subsets"
)

SYNTHETIC_DATASET_DIR = (
    Path("../data/processed/synthetic_subsets")
    / (
        "10_percent_signal_augmentation_"
        "ratio_1_seed_42"
    )
)

MULTISEED_OUTPUT_DIR = Path(
    "../outputs/"
    "multiseed_augmentation_robustness"
)

MULTISEED_CHECKPOINT_DIR = Path(
    "../checkpoints/"
    "multiseed_augmentation_robustness"
)

RATIO_OUTPUT_DIR = Path(
    "../outputs/"
    "synthetic_ratio_ablation"
)

RATIO_CHECKPOINT_DIR = Path(
    "../checkpoints/"
    "synthetic_ratio_ablation"
)

SELECTION_INDEX_PATH = (
    RATIO_OUTPUT_DIR
    / "synthetic_ratio_0_5_indices.npy"
)

RATIO_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RATIO_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

required_source_files = [
    OFFICIAL_DATA_DIR / "X_train.npy",
    OFFICIAL_DATA_DIR / "y_train.npy",
    OFFICIAL_DATA_DIR / "X_validation.npy",
    OFFICIAL_DATA_DIR / "y_validation.npy",
    OFFICIAL_DATA_DIR / "X_test.npy",
    OFFICIAL_DATA_DIR / "y_test.npy",

    LIMITED_DATA_DIR
    / "indices_10_percent.npy",

    SYNTHETIC_DATASET_DIR
    / "X_synthetic.npy",

    SYNTHETIC_DATASET_DIR
    / "y_synthetic.npy",

    MULTISEED_OUTPUT_DIR
    / "all_seed_test_metrics.csv",

    MULTISEED_OUTPUT_DIR
    / "paired_improvement_summary.csv"
]

missing_source_files = [
    path
    for path in required_source_files
    if not path.exists()
]

if missing_source_files:
    raise FileNotFoundError(
        "Missing required source files:\n"
        + "\n".join(
            str(path.resolve())
            for path in missing_source_files
        )
    )

print(
    "All required source files were found."
)

print(
    "Ratio output directory:",
    RATIO_OUTPUT_DIR.resolve()
)

print(
    "Ratio checkpoint directory:",
    RATIO_CHECKPOINT_DIR.resolve()
)

## 2. Deterministic 0.5:1 Synthetic Subset

The reduced synthetic subset contains 288 bird and 288 drone observations. The subset is selected once with seed 42 and reused across all five model-training seeds, ensuring that the experiment measures training stochasticity rather than synthetic-subset variability.

In [ ]:
X_train_complete = np.load(
    OFFICIAL_DATA_DIR / "X_train.npy",
    mmap_mode="r"
)

y_train_complete = np.load(
    OFFICIAL_DATA_DIR / "y_train.npy",
    mmap_mode="r"
)

real_source_indices = np.load(
    LIMITED_DATA_DIR
    / "indices_10_percent.npy"
)

X_real = np.asarray(
    X_train_complete[
        real_source_indices
    ],
    dtype=np.float32
)

y_real = np.asarray(
    y_train_complete[
        real_source_indices
    ],
    dtype=np.uint8
)

X_synthetic_complete = np.asarray(
    np.load(
        SYNTHETIC_DATASET_DIR
        / "X_synthetic.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_synthetic_complete = np.asarray(
    np.load(
        SYNTHETIC_DATASET_DIR
        / "y_synthetic.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

X_validation = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "X_validation.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_validation = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "y_validation.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

X_test = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "X_test.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_test = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "y_test.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

print("Real tensor:", X_real.shape)

print(
    "Complete synthetic tensor:",
    X_synthetic_complete.shape
)

print(
    "Validation tensor:",
    X_validation.shape
)

print("Test tensor:", X_test.shape)

In [ ]:
selection_rng = np.random.default_rng(
    SYNTHETIC_SELECTION_SEED
)

selected_indices_by_class = []

for class_label in [
    0,
    1
]:
    class_indices = np.flatnonzero(
        y_synthetic_complete
        == class_label
    )

    selected_class_indices = (
        selection_rng.choice(
            class_indices,
            size=(
                HALF_RATIO_SYNTHETIC_PER_CLASS
            ),
            replace=False
        )
    )

    selected_indices_by_class.append(
        np.sort(
            selected_class_indices
        )
    )

generated_selection_indices = (
    np.sort(
        np.concatenate(
            selected_indices_by_class
        )
    )
)

if SELECTION_INDEX_PATH.exists():
    saved_selection_indices = np.load(
        SELECTION_INDEX_PATH
    )

    assert np.array_equal(
        saved_selection_indices,
        generated_selection_indices
    )

    synthetic_half_indices = (
        saved_selection_indices
    )

    print(
        "Existing deterministic selection "
        "indices verified."
    )

else:
    np.save(
        SELECTION_INDEX_PATH,
        generated_selection_indices
    )

    synthetic_half_indices = (
        generated_selection_indices
    )

    print(
        "Deterministic selection indices "
        "saved."
    )

X_synthetic_half = (
    X_synthetic_complete[
        synthetic_half_indices
    ]
)

y_synthetic_half = (
    y_synthetic_complete[
        synthetic_half_indices
    ]
)

print(
    "Reduced synthetic tensor:",
    X_synthetic_half.shape
)

print(
    "Reduced synthetic class counts:",
    np.bincount(
        y_synthetic_half,
        minlength=2
    )
)

In [ ]:
X_ratio_training = np.concatenate(
    [
        X_real,
        X_synthetic_half
    ],
    axis=0
)

y_ratio_training = np.concatenate(
    [
        y_real,
        y_synthetic_half
    ],
    axis=0
)

assert X_real.shape == (
    1150,
    5,
    150
)

assert np.array_equal(
    np.bincount(
        y_real,
        minlength=2
    ),
    [575, 575]
)

assert X_synthetic_half.shape == (
    576,
    5,
    150
)

assert np.array_equal(
    np.bincount(
        y_synthetic_half,
        minlength=2
    ),
    [288, 288]
)

assert X_ratio_training.shape == (
    1726,
    5,
    150
)

assert np.array_equal(
    np.bincount(
        y_ratio_training,
        minlength=2
    ),
    [863, 863]
)

for dataset_name, dataset in {
    "X_real": X_real,
    "X_synthetic_half":
        X_synthetic_half,
    "X_ratio_training":
        X_ratio_training,
    "X_validation":
        X_validation,
    "X_test":
        X_test
}.items():
    assert dataset.dtype == np.float32
    assert np.isfinite(dataset).all()
    assert dataset.min() >= 0.0
    assert dataset.max() <= 1.0

print(
    "0.5:1 training tensor:",
    X_ratio_training.shape
)

print(
    "0.5:1 class counts:",
    np.bincount(
        y_ratio_training,
        minlength=2
    )
)

print(
    "The deterministic 0.5:1 dataset "
    "passed all validation checks."
)

In [ ]:
ratio_run_status_records = []

for model_seed in MODEL_SEEDS:
    run_name = (
        f"{RATIO_CONFIGURATION}_"
        f"seed_{model_seed}"
    )

    output_directory = (
        RATIO_OUTPUT_DIR
        / run_name
    )

    checkpoint_path = (
        RATIO_CHECKPOINT_DIR
        / f"{run_name}.keras"
    )

    history_path = (
        output_directory
        / "training_history.csv"
    )

    validation_path = (
        output_directory
        / "validation_results.csv"
    )

    threshold_path = (
        output_directory
        / "validation_threshold_search.csv"
    )

    test_path = (
        output_directory
        / "test_metrics.csv"
    )

    checkpoint_exists = (
        checkpoint_path.exists()
    )

    history_exists = (
        history_path.exists()
    )

    validation_exists = (
        validation_path.exists()
    )

    threshold_exists = (
        threshold_path.exists()
    )

    test_exists = (
        test_path.exists()
    )

    training_complete = (
        checkpoint_exists
        and history_exists
    )

    evaluation_complete = (
        validation_exists
        and threshold_exists
    )

    inconsistent_artifacts = (
        checkpoint_exists
        != history_exists
    ) or (
        validation_exists
        != threshold_exists
    ) or (
        evaluation_complete
        and not training_complete
    ) or (
        test_exists
    )

    if inconsistent_artifacts:
        status = "inconsistent_artifacts"
    elif (
        training_complete
        and evaluation_complete
    ):
        status = "complete"
    elif training_complete:
        status = (
            "trained_awaiting_evaluation"
        )
    else:
        status = "not_started"

    ratio_run_status_records.append({
        "seed":
            model_seed,

        "status":
            status,

        "checkpoint_exists":
            checkpoint_exists,

        "history_exists":
            history_exists,

        "validation_exists":
            validation_exists,

        "threshold_search_exists":
            threshold_exists,

        "test_exists":
            test_exists,

        "training_complete":
            training_complete,

        "run_complete":
            (
                training_complete
                and evaluation_complete
            ),

        "output_directory":
            str(output_directory)
    })

ratio_run_status_df = pd.DataFrame(
    ratio_run_status_records
)

display(
    ratio_run_status_df
)

inconsistent_ratio_runs = (
    ratio_run_status_df[
        ratio_run_status_df["status"]
        == "inconsistent_artifacts"
    ]
)

if not inconsistent_ratio_runs.empty:
    raise RuntimeError(
        "Inconsistent ratio-experiment "
        "artifacts were found."
    )

if RUN_RATIO_TRAINING:
    print(
        "Training mode requested."
    )
else:
    print(
        "SAFE INSPECTION MODE: ratio "
        "training is disabled."
    )

## 3. Fixed CNN and Protected Training

The five 0.5:1 models use the same 29,121-parameter CNN, optimizer, callbacks, and training procedure as the previous experiments. Each seed has independent protected output and checkpoint paths.

In [ ]:
X_ratio_training_model = (
    X_ratio_training[
        ...,
        np.newaxis
    ]
)

X_validation_model = (
    X_validation[
        ...,
        np.newaxis
    ]
)

X_test_model = (
    X_test[
        ...,
        np.newaxis
    ]
)

def build_ratio_datasets(
    model_seed
):
    training_dataset = (
        tf.data.Dataset
        .from_tensor_slices((
            X_ratio_training_model,
            y_ratio_training
        ))
        .shuffle(
            buffer_size=len(
                y_ratio_training
            ),
            seed=model_seed,
            reshuffle_each_iteration=True
        )
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    validation_dataset = (
        tf.data.Dataset
        .from_tensor_slices((
            X_validation_model,
            y_validation
        ))
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    test_dataset = (
        tf.data.Dataset
        .from_tensor_slices((
            X_test_model,
            y_test
        ))
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    return (
        training_dataset,
        validation_dataset,
        test_dataset
    )


(
    example_training_dataset,
    example_validation_dataset,
    example_test_dataset
) = build_ratio_datasets(
    MODEL_SEEDS[0]
)

print(
    "Training batches:",
    len(example_training_dataset)
)

print(
    "Validation batches:",
    len(example_validation_dataset)
)

print(
    "Test batches:",
    len(example_test_dataset)
)

In [ ]:
def build_fixed_cnn(
    input_shape=(5, 150, 1)
):
    model = models.Sequential([
        layers.Input(
            shape=input_shape
        ),

        layers.Conv2D(
            16,
            kernel_size=(3, 7),
            padding="same",
            use_bias=True
        ),

        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.Conv2D(
            32,
            kernel_size=(3, 5),
            padding="same",
            use_bias=True
        ),

        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.Conv2D(
            64,
            kernel_size=(3, 3),
            padding="same",
            use_bias=True
        ),

        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            32,
            activation="relu",
            kernel_regularizer=(
                regularizers.l2(1e-4)
            )
        ),

        layers.Dropout(0.30),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),

        loss="binary_crossentropy",

        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),

            tf.keras.metrics.AUC(
                name="roc_auc"
            ),

            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR"
            ),

            tf.keras.metrics.Precision(
                name="precision"
            ),

            tf.keras.metrics.Recall(
                name="recall"
            )
        ]
    )

    return model

In [ ]:
def set_model_seed(model_seed):
    tf.keras.backend.clear_session()

    random.seed(model_seed)
    np.random.seed(model_seed)
    tf.random.set_seed(model_seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def get_ratio_run_paths(model_seed):
    run_name = (
        f"{RATIO_CONFIGURATION}_"
        f"seed_{model_seed}"
    )

    output_directory = (
        RATIO_OUTPUT_DIR
        / run_name
    )

    return {
        "run_name":
            run_name,

        "output_directory":
            output_directory,

        "checkpoint":
            RATIO_CHECKPOINT_DIR
            / f"{run_name}.keras",

        "history":
            output_directory
            / "training_history.csv",

        "training_log":
            output_directory
            / "training_log.csv",

        "configuration":
            output_directory
            / "run_configuration.json",

        "validation_results":
            output_directory
            / "validation_results.csv",

        "threshold_search":
            output_directory
            / "validation_threshold_search.csv",

        "test_metrics":
            output_directory
            / "test_metrics.csv",

        "test_predictions":
            output_directory
            / "test_predictions.csv"
    }

In [ ]:
set_model_seed(
    MODEL_SEEDS[0]
)

verification_model = (
    build_fixed_cnn()
)

assert (
    verification_model.count_params()
    == 29121
)

print(
    "Architecture parameters:",
    verification_model.count_params()
)

print(
    "Architecture verified against "
    "all previous experiments."
)

verification_model.summary()

del verification_model

tf.keras.backend.clear_session()

In [ ]:
def train_ratio_run(model_seed):
    run_paths = get_ratio_run_paths(
        model_seed
    )

    checkpoint_exists = (
        run_paths["checkpoint"].exists()
    )

    history_exists = (
        run_paths["history"].exists()
    )

    if (
        checkpoint_exists
        and history_exists
    ):
        print(
            "Training already complete; "
            "skipping:",
            run_paths["run_name"]
        )

        return {
            "seed":
                model_seed,

            "status":
                "already_trained"
        }

    if (
        checkpoint_exists
        != history_exists
    ):
        raise RuntimeError(
            "Inconsistent training artifacts "
            "for "
            + run_paths["run_name"]
        )

    unexpected_existing_files = [
        path
        for path in [
            run_paths["training_log"],
            run_paths["configuration"],
            run_paths["validation_results"],
            run_paths["threshold_search"],
            run_paths["test_metrics"],
            run_paths["test_predictions"]
        ]
        if path.exists()
    ]

    if unexpected_existing_files:
        raise FileExistsError(
            "Unexpected existing artifacts "
            "for "
            + run_paths["run_name"]
            + ":\n"
            + "\n".join(
                str(path.resolve())
                for path
                in unexpected_existing_files
            )
        )

    run_paths["output_directory"].mkdir(
        parents=True,
        exist_ok=True
    )

    set_model_seed(
        model_seed
    )

    (
        training_dataset,
        validation_dataset,
        _
    ) = build_ratio_datasets(
        model_seed
    )

    model = build_fixed_cnn()

    assert model.count_params() == 29121

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=run_paths[
                "checkpoint"
            ],
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        ),

        tf.keras.callbacks.CSVLogger(
            run_paths[
                "training_log"
            ]
        )
    ]

    print()
    print("=" * 70)

    print(
        "Training:",
        run_paths["run_name"]
    )

    print(
        "Training observations:",
        len(y_ratio_training)
    )

    print("=" * 70)

    history = model.fit(
        training_dataset,
        validation_data=(
            validation_dataset
        ),
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        shuffle=False,
        verbose=1
    )

    history_df = pd.DataFrame(
        history.history
    )

    history_df.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(history_df) + 1
        )
    )

    history_df.to_csv(
        run_paths["history"],
        index=False
    )

    best_history_index = (
        history_df[
            "val_loss"
        ].idxmin()
    )

    best_epoch = int(
        history_df.loc[
            best_history_index,
            "epoch"
        ]
    )

    best_validation_loss = float(
        history_df.loc[
            best_history_index,
            "val_loss"
        ]
    )

    run_configuration = {
        "configuration":
            RATIO_CONFIGURATION,

        "model_seed":
            model_seed,

        "synthetic_selection_seed":
            SYNTHETIC_SELECTION_SEED,

        "real_observations":
            int(len(y_real)),

        "synthetic_observations":
            int(
                len(y_synthetic_half)
            ),

        "total_training_observations":
            int(
                len(y_ratio_training)
            ),

        "target_synthetic_ratio":
            TARGET_SYNTHETIC_RATIO,

        "actual_synthetic_ratio":
            ACTUAL_SYNTHETIC_RATIO,

        "synthetic_samples_per_class":
            HALF_RATIO_SYNTHETIC_PER_CLASS,

        "batch_size":
            BATCH_SIZE,

        "maximum_epochs":
            MAX_EPOCHS,

        "completed_epochs":
            int(len(history_df)),

        "best_epoch":
            best_epoch,

        "best_validation_loss":
            best_validation_loss,

        "model_parameters":
            int(
                model.count_params()
            )
    }

    with open(
        run_paths["configuration"],
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            run_configuration,
            file,
            indent=2
        )

    assert (
        run_paths["checkpoint"].exists()
    )

    assert (
        run_paths["history"].exists()
    )

    print(
        "Completed:",
        run_paths["run_name"]
    )

    print(
        "Completed epochs:",
        len(history_df)
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "Best validation loss:",
        best_validation_loss
    )

    del model
    del history

    tf.keras.backend.clear_session()

    return {
        "seed":
            model_seed,

        "status":
            "trained",

        "completed_epochs":
            int(len(history_df)),

        "best_epoch":
            best_epoch,

        "best_validation_loss":
            best_validation_loss
    }

In [ ]:
pending_ratio_records = []

for model_seed in MODEL_SEEDS:
    run_paths = get_ratio_run_paths(
        model_seed
    )

    training_complete = (
        run_paths["checkpoint"].exists()
        and run_paths["history"].exists()
    )

    pending_ratio_records.append({
        "seed":
            model_seed,

        "training_observations":
            len(y_ratio_training),

        "real_observations":
            len(y_real),

        "synthetic_observations":
            len(y_synthetic_half),

        "training_complete":
            training_complete,

        "action":
            (
                "skip"
                if training_complete
                else "train"
            )
    })

pending_ratio_df = pd.DataFrame(
    pending_ratio_records
)

display(
    pending_ratio_df
)

print(
    "Runs requiring training:",
    int(
        (
            pending_ratio_df["action"]
            == "train"
        ).sum()
    )
)

In [ ]:
if not RUN_RATIO_TRAINING:
    print(
        "Ratio training remains disabled."
    )

else:
    ratio_training_records = []

    for model_seed in MODEL_SEEDS:
        training_result = (
            train_ratio_run(
                model_seed
            )
        )

        ratio_training_records.append(
            training_result
        )

    ratio_training_summary_df = (
        pd.DataFrame(
            ratio_training_records
        )
    )

    display(
        ratio_training_summary_df
    )

    RUN_RATIO_TRAINING = False

    print(
        "All requested ratio-training "
        "calls finished."
    )

    print(
        "RUN_RATIO_TRAINING was reset "
        "to False in memory."
    )

## 4. Validation-Only Ratio Comparison

The 0:1, 0.5:1, and 1:1 configurations are compared across the same five training seeds using only the official validation partition. The test set remains untouched during ratio selection.

In [ ]:
BASELINE_CHECKPOINT_DIR = Path(
    "../checkpoints/baseline"
)

AUGMENTED_CHECKPOINT_DIR = Path(
    "../checkpoints/"
    "synthetic_augmentation"
)

ratio_configurations = {
    0.0: {
        "name":
            "real_only",

        "display_name":
            "0:1 — real-only"
    },

    0.5: {
        "name":
            RATIO_CONFIGURATION,

        "display_name":
            "0.5:1 — reduced augmentation"
    },

    1.0: {
        "name":
            "real_plus_synthetic",

        "display_name":
            "1:1 — full augmentation"
    }
}

def get_comparison_checkpoint(
    synthetic_ratio,
    model_seed
):
    if synthetic_ratio == 0.0:
        if model_seed == 42:
            return (
                BASELINE_CHECKPOINT_DIR
                / (
                    "baseline_10_percent_"
                    "seed_42.keras"
                )
            )

        return (
            MULTISEED_CHECKPOINT_DIR
            / (
                f"real_only_seed_"
                f"{model_seed}.keras"
            )
        )

    if synthetic_ratio == 0.5:
        return (
            get_ratio_run_paths(
                model_seed
            )["checkpoint"]
        )

    if synthetic_ratio == 1.0:
        if model_seed == 42:
            return (
                AUGMENTED_CHECKPOINT_DIR
                / (
                    "10_percent_real_plus_"
                    "synthetic_1_to_1_"
                    "seed_42.keras"
                )
            )

        return (
            MULTISEED_CHECKPOINT_DIR
            / (
                "real_plus_synthetic_"
                f"seed_{model_seed}.keras"
            )
        )

    raise ValueError(
        "Unsupported synthetic ratio: "
        + str(synthetic_ratio)
    )

In [ ]:
checkpoint_inventory_records = []

for synthetic_ratio in [
    0.0,
    0.5,
    1.0
]:
    for model_seed in MODEL_SEEDS:
        checkpoint_path = (
            get_comparison_checkpoint(
                synthetic_ratio,
                model_seed
            )
        )

        checkpoint_inventory_records.append({
            "synthetic_ratio":
                synthetic_ratio,

            "configuration":
                ratio_configurations[
                    synthetic_ratio
                ]["display_name"],

            "seed":
                model_seed,

            "checkpoint_exists":
                checkpoint_path.exists(),

            "checkpoint_path":
                str(checkpoint_path)
        })

checkpoint_inventory_df = pd.DataFrame(
    checkpoint_inventory_records
)

display(
    checkpoint_inventory_df
)

missing_checkpoints = (
    checkpoint_inventory_df[
        ~checkpoint_inventory_df[
            "checkpoint_exists"
        ]
    ]
)

if not missing_checkpoints.empty:
    raise FileNotFoundError(
        "Missing comparison checkpoints:\n"
        + missing_checkpoints[
            [
                "synthetic_ratio",
                "seed",
                "checkpoint_path"
            ]
        ].to_string(index=False)
    )

print(
    "All 15 ratio-comparison "
    "checkpoints were found."
)

In [ ]:
def calculate_validation_metrics(
    true_labels,
    probabilities,
    threshold
):
    predictions = (
        probabilities
        >= threshold
    ).astype(np.uint8)

    return {
        "accuracy":
            accuracy_score(
                true_labels,
                predictions
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                true_labels,
                predictions
            ),

        "macro_f1":
            f1_score(
                true_labels,
                predictions,
                average="macro",
                zero_division=0
            ),

        "bird_precision":
            precision_score(
                true_labels,
                predictions,
                pos_label=0,
                zero_division=0
            ),

        "bird_recall":
            recall_score(
                true_labels,
                predictions,
                pos_label=0,
                zero_division=0
            ),

        "bird_f1":
            f1_score(
                true_labels,
                predictions,
                pos_label=0,
                zero_division=0
            ),

        "drone_recall":
            recall_score(
                true_labels,
                predictions,
                pos_label=1,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                true_labels,
                probabilities
            )
    }

In [ ]:
def evaluate_checkpoint_on_validation(
    synthetic_ratio,
    model_seed
):
    checkpoint_path = (
        get_comparison_checkpoint(
            synthetic_ratio,
            model_seed
        )
    )

    set_model_seed(
        model_seed
    )

    model = tf.keras.models.load_model(
        checkpoint_path
    )

    assert model.count_params() == 29121

    (
        _,
        validation_dataset,
        _
    ) = build_ratio_datasets(
        model_seed
    )

    validation_probabilities = (
        model.predict(
            validation_dataset,
            verbose=0
        )
        .reshape(-1)
    )

    assert (
        validation_probabilities.shape
        == y_validation.shape
    )

    assert np.isfinite(
        validation_probabilities
    ).all()

    threshold_records = []

    for threshold in np.linspace(
        0.01,
        0.99,
        199
    ):
        threshold_metrics = (
            calculate_validation_metrics(
                y_validation,
                validation_probabilities,
                threshold
            )
        )

        threshold_records.append({
            "threshold":
                float(threshold),

            **threshold_metrics
        })

    threshold_search_df = pd.DataFrame(
        threshold_records
    )

    optimal_row = (
        threshold_search_df
        .sort_values(
            [
                "macro_f1",
                "balanced_accuracy",
                "accuracy"
            ],
            ascending=[
                False,
                False,
                False
            ],
            kind="mergesort"
        )
        .iloc[0]
    )

    result = {
        "synthetic_ratio":
            synthetic_ratio,

        "configuration":
            ratio_configurations[
                synthetic_ratio
            ]["name"],

        "configuration_display_name":
            ratio_configurations[
                synthetic_ratio
            ]["display_name"],

        "seed":
            model_seed,

        "threshold":
            float(
                optimal_row[
                    "threshold"
                ]
            ),

        "validation_accuracy":
            float(
                optimal_row[
                    "accuracy"
                ]
            ),

        "validation_balanced_accuracy":
            float(
                optimal_row[
                    "balanced_accuracy"
                ]
            ),

        "validation_macro_f1":
            float(
                optimal_row[
                    "macro_f1"
                ]
            ),

        "validation_bird_precision":
            float(
                optimal_row[
                    "bird_precision"
                ]
            ),

        "validation_bird_recall":
            float(
                optimal_row[
                    "bird_recall"
                ]
            ),

        "validation_bird_f1":
            float(
                optimal_row[
                    "bird_f1"
                ]
            ),

        "validation_drone_recall":
            float(
                optimal_row[
                    "drone_recall"
                ]
            ),

        "validation_roc_auc":
            float(
                optimal_row[
                    "roc_auc"
                ]
            )
    }

    if synthetic_ratio == 0.5:
        run_paths = get_ratio_run_paths(
            model_seed
        )

        validation_path = (
            run_paths[
                "validation_results"
            ]
        )

        threshold_path = (
            run_paths[
                "threshold_search"
            ]
        )

        if validation_path.exists():
            existing_validation = (
                pd.read_csv(
                    validation_path
                )
            )

            assert np.isclose(
                existing_validation.loc[
                    0,
                    "threshold"
                ],
                result["threshold"]
            )

            print(
                "Existing 0.5:1 validation "
                "result verified:",
                model_seed
            )

        else:
            pd.DataFrame([
                result
            ]).to_csv(
                validation_path,
                index=False
            )

            threshold_search_df.to_csv(
                threshold_path,
                index=False
            )

    del model

    tf.keras.backend.clear_session()

    return result

In [ ]:
assert RUN_RATIO_TRAINING is False

validation_ratio_records = []

for synthetic_ratio in [
    0.0,
    0.5,
    1.0
]:
    for model_seed in MODEL_SEEDS:
        validation_result = (
            evaluate_checkpoint_on_validation(
                synthetic_ratio,
                model_seed
            )
        )

        validation_ratio_records.append(
            validation_result
        )

        print(
            "Validated:",
            synthetic_ratio,
            "seed",
            model_seed,
            "macro-F1:",
            round(
                validation_result[
                    "validation_macro_f1"
                ],
                4
            )
        )

validation_ratio_df = (
    pd.DataFrame(
        validation_ratio_records
    )
    .sort_values([
        "synthetic_ratio",
        "seed"
    ])
    .reset_index(drop=True)
)

assert len(validation_ratio_df) == 15

validation_ratio_df.to_csv(
    RATIO_OUTPUT_DIR
    / "validation_ratio_results.csv",
    index=False
)

display(
    validation_ratio_df.style.format({
        "synthetic_ratio": "{:.1f}",
        "threshold": "{:.4f}",
        "validation_accuracy": "{:.4f}",
        "validation_balanced_accuracy":
            "{:.4f}",
        "validation_macro_f1": "{:.4f}",
        "validation_bird_precision":
            "{:.4f}",
        "validation_bird_recall":
            "{:.4f}",
        "validation_bird_f1": "{:.4f}",
        "validation_drone_recall":
            "{:.4f}",
        "validation_roc_auc": "{:.4f}"
    })
)

In [ ]:
validation_summary_columns = [
    "validation_accuracy",
    "validation_balanced_accuracy",
    "validation_macro_f1",
    "validation_bird_precision",
    "validation_bird_recall",
    "validation_bird_f1",
    "validation_drone_recall",
    "validation_roc_auc"
]

validation_ratio_summary_df = (
    validation_ratio_df
    .groupby(
        [
            "synthetic_ratio",
            "configuration_display_name"
        ],
        observed=True
    )[validation_summary_columns]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
)

validation_ratio_summary_df.columns = [
    f"{metric}_{statistic}"
    for metric, statistic
    in validation_ratio_summary_df.columns
]

validation_ratio_summary_df = (
    validation_ratio_summary_df
    .reset_index()
)

validation_ratio_summary_df.to_csv(
    RATIO_OUTPUT_DIR
    / "validation_ratio_summary.csv",
    index=False
)

ratio_selection_table = (
    validation_ratio_summary_df
    .sort_values(
        [
            "validation_macro_f1_mean",
            "validation_balanced_accuracy_mean",
            "validation_accuracy_mean"
        ],
        ascending=[
            False,
            False,
            False
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

SELECTED_SYNTHETIC_RATIO = float(
    ratio_selection_table.loc[
        0,
        "synthetic_ratio"
    ]
)

display(
    validation_ratio_summary_df.style.format({
        column: "{:.4f}"
        for column
        in validation_ratio_summary_df.columns
        if column not in [
            "synthetic_ratio",
            "configuration_display_name"
        ]
    })
)

print(
    "Selected synthetic ratio:",
    SELECTED_SYNTHETIC_RATIO
)

print(
    "Selection criterion: highest mean "
    "validation macro-F1, followed by "
    "mean balanced accuracy and accuracy."
)

In [ ]:
validation_plot_df = (
    validation_ratio_df[
        [
            "synthetic_ratio",
            "seed",
            "validation_balanced_accuracy",
            "validation_macro_f1",
            "validation_bird_f1",
            "validation_roc_auc"
        ]
    ]
    .melt(
        id_vars=[
            "synthetic_ratio",
            "seed"
        ],
        var_name="metric",
        value_name="score"
    )
)

validation_metric_names = {
    "validation_balanced_accuracy":
        "Balanced accuracy",

    "validation_macro_f1":
        "Macro-F1",

    "validation_bird_f1":
        "Bird F1",

    "validation_roc_auc":
        "ROC-AUC"
}

validation_plot_df["metric"] = (
    validation_plot_df["metric"]
    .map(
        validation_metric_names
    )
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
    sharex=True,
    constrained_layout=True
)

for axis, metric_name in zip(
    axes.flat,
    validation_metric_names.values()
):
    metric_subset = (
        validation_plot_df[
            validation_plot_df["metric"]
            == metric_name
        ]
    )

    sns.lineplot(
        data=metric_subset,
        x="synthetic_ratio",
        y="score",
        hue="seed",
        marker="o",
        linewidth=1.8,
        ax=axis
    )

    axis.set_title(metric_name)
    axis.set_xlabel(
        "Synthetic-to-real ratio"
    )

    axis.set_ylabel(
        "Validation score"
    )

    axis.set_xticks([
        0.0,
        0.5,
        1.0
    ])

    axis.set_ylim(
        0.0,
        1.02
    )

    axis.grid(alpha=0.25)

    if axis is not axes[0, 0]:
        axis.get_legend().remove()

axes[0, 0].legend(
    title="Training seed",
    loc="lower right"
)

fig.suptitle(
    "Validation Performance across "
    "Synthetic-to-Real Ratios",
    fontsize=14
)

plt.show()

## 5. Incremental Benefit and Data Efficiency

This section quantifies how much of the improvement from 0:1 to 1:1 is already recovered by the 0.5:1 ratio, and whether increasing from 0.5:1 to 1:1 improves each seed consistently.

In [ ]:
ratio_efficiency_metrics = {
    "validation_accuracy":
        "Accuracy",

    "validation_balanced_accuracy":
        "Balanced accuracy",

    "validation_macro_f1":
        "Macro-F1",

    "validation_bird_f1":
        "Bird F1",

    "validation_drone_recall":
        "Drone recall",

    "validation_roc_auc":
        "ROC-AUC"
}

ratio_efficiency_records = []

ratio_seed_difference_records = []

for (
    metric_column,
    metric_display_name
) in ratio_efficiency_metrics.items():
    metric_pivot = (
        validation_ratio_df
        .pivot(
            index="seed",
            columns="synthetic_ratio",
            values=metric_column
        )
        .reindex(MODEL_SEEDS)
    )

    metric_pivot.columns = [
        float(column)
        for column in metric_pivot.columns
    ]

    half_gain = (
        metric_pivot[0.5]
        - metric_pivot[0.0]
    )

    full_gain = (
        metric_pivot[1.0]
        - metric_pivot[0.0]
    )

    incremental_full_gain = (
        metric_pivot[1.0]
        - metric_pivot[0.5]
    )

    mean_half_gain = float(
        half_gain.mean()
    )

    mean_full_gain = float(
        full_gain.mean()
    )

    if not np.isclose(
        mean_full_gain,
        0.0
    ):
        recovered_benefit_percent = (
            mean_half_gain
            / mean_full_gain
            * 100.0
        )
    else:
        recovered_benefit_percent = (
            np.nan
        )

    ratio_efficiency_records.append({
        "metric":
            metric_column,

        "metric_display_name":
            metric_display_name,

        "real_only_mean":
            float(
                metric_pivot[0.0].mean()
            ),

        "half_ratio_mean":
            float(
                metric_pivot[0.5].mean()
            ),

        "full_ratio_mean":
            float(
                metric_pivot[1.0].mean()
            ),

        "half_ratio_gain_over_real":
            mean_half_gain,

        "full_ratio_gain_over_real":
            mean_full_gain,

        "incremental_gain_full_vs_half":
            float(
                incremental_full_gain.mean()
            ),

        "benefit_recovered_by_half_percent":
            recovered_benefit_percent,

        "full_ratio_seed_wins_vs_half":
            int(
                np.sum(
                    incremental_full_gain
                    > 0
                )
            ),

        "ties":
            int(
                np.sum(
                    np.isclose(
                        incremental_full_gain,
                        0.0
                    )
                )
            ),

        "full_ratio_seed_losses_vs_half":
            int(
                np.sum(
                    incremental_full_gain
                    < 0
                )
            )
    })

    for model_seed in MODEL_SEEDS:
        ratio_seed_difference_records.append({
            "seed":
                model_seed,

            "metric":
                metric_column,

            "half_ratio_minus_real":
                float(
                    half_gain.loc[
                        model_seed
                    ]
                ),

            "full_ratio_minus_real":
                float(
                    full_gain.loc[
                        model_seed
                    ]
                ),

            "full_ratio_minus_half":
                float(
                    incremental_full_gain.loc[
                        model_seed
                    ]
                )
        })

ratio_efficiency_df = pd.DataFrame(
    ratio_efficiency_records
)

ratio_seed_differences_df = pd.DataFrame(
    ratio_seed_difference_records
)

ratio_efficiency_df.to_csv(
    RATIO_OUTPUT_DIR
    / "ratio_efficiency_summary.csv",
    index=False
)

ratio_seed_differences_df.to_csv(
    RATIO_OUTPUT_DIR
    / "ratio_seed_differences.csv",
    index=False
)

display(
    ratio_efficiency_df.style.format({
        "real_only_mean": "{:.4f}",
        "half_ratio_mean": "{:.4f}",
        "full_ratio_mean": "{:.4f}",
        "half_ratio_gain_over_real":
            "{:+.4f}",
        "full_ratio_gain_over_real":
            "{:+.4f}",
        "incremental_gain_full_vs_half":
            "{:+.4f}",
        "benefit_recovered_by_half_percent":
            "{:.2f}%"
    })
)

In [ ]:
summary_plot_records = []

for synthetic_ratio in [
    0.0,
    0.5,
    1.0
]:
    ratio_rows = (
        validation_ratio_df[
            validation_ratio_df[
                "synthetic_ratio"
            ] == synthetic_ratio
        ]
    )

    for (
        metric_column,
        metric_display_name
    ) in ratio_efficiency_metrics.items():
        summary_plot_records.append({
            "synthetic_ratio":
                synthetic_ratio,

            "metric":
                metric_display_name,

            "mean":
                float(
                    ratio_rows[
                        metric_column
                    ].mean()
                ),

            "standard_deviation":
                float(
                    ratio_rows[
                        metric_column
                    ].std(ddof=1)
                )
        })

ratio_mean_plot_df = pd.DataFrame(
    summary_plot_records
)

selected_plot_metrics = [
    "Balanced accuracy",
    "Macro-F1",
    "Bird F1",
    "ROC-AUC"
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
    sharex=True,
    constrained_layout=True
)

for axis, metric_name in zip(
    axes.flat,
    selected_plot_metrics
):
    metric_rows = (
        ratio_mean_plot_df[
            ratio_mean_plot_df["metric"]
            == metric_name
        ]
    )

    axis.errorbar(
        metric_rows[
            "synthetic_ratio"
        ],
        metric_rows["mean"],
        yerr=metric_rows[
            "standard_deviation"
        ],
        marker="o",
        linewidth=2,
        capsize=5
    )

    axis.set_title(metric_name)

    axis.set_xlabel(
        "Synthetic-to-real ratio"
    )

    axis.set_ylabel(
        "Mean validation score ± SD"
    )

    axis.set_xticks([
        0.0,
        0.5,
        1.0
    ])

    axis.set_ylim(
        0.0,
        1.02
    )

    axis.grid(alpha=0.25)

fig.suptitle(
    "Validation Performance and "
    "Variability by Synthetic Ratio",
    fontsize=14
)

plt.show()

In [ ]:
assert np.isclose(
    SELECTED_SYNTHETIC_RATIO,
    1.0
)

selected_ratio_row = (
    validation_ratio_summary_df[
        np.isclose(
            validation_ratio_summary_df[
                "synthetic_ratio"
            ],
            SELECTED_SYNTHETIC_RATIO
        )
    ]
    .iloc[0]
)

ratio_selection_manifest = {
    "experiment":
        "synthetic_ratio_ablation",

    "candidate_ratios": [
        0.0,
        0.5,
        1.0
    ],

    "model_seeds":
        MODEL_SEEDS,

    "synthetic_selection_seed":
        SYNTHETIC_SELECTION_SEED,

    "selected_synthetic_ratio":
        SELECTED_SYNTHETIC_RATIO,

    "selection_partition":
        "validation",

    "selection_primary_metric":
        "mean_macro_f1",

    "selection_tie_breakers": [
        "mean_balanced_accuracy",
        "mean_accuracy"
    ],

    "selected_mean_validation_macro_f1":
        float(
            selected_ratio_row[
                "validation_macro_f1_mean"
            ]
        ),

    "selected_mean_validation_balanced_accuracy":
        float(
            selected_ratio_row[
                "validation_balanced_accuracy_mean"
            ]
        ),

    "selected_mean_validation_accuracy":
        float(
            selected_ratio_row[
                "validation_accuracy_mean"
            ]
        ),

    "new_test_evaluation_performed":
        False,

    "reason_no_new_test_evaluation":
        (
            "The selected 1:1 configuration "
            "was already evaluated on the "
            "held-out test set in Notebook 09."
        )
}

with open(
    RATIO_OUTPUT_DIR
    / "ratio_selection_manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        ratio_selection_manifest,
        file,
        indent=2
    )

print(
    "Locked synthetic ratio:",
    SELECTED_SYNTHETIC_RATIO
)

print(
    "No new test inference was performed."
)

print(
    "Ratio-selection manifest saved."
)

## 6. Discussion, Limitations, and Conclusion

### Validation-based ratio selection

The synthetic-to-real ratio was selected using mean validation macro-F1 across five paired training seeds. The 1:1 configuration achieved the highest mean validation macro-F1 (**0.9219**), followed by 0.5:1 (**0.8950**) and real-only training (**0.8041**).

The same ordering was observed for mean balanced accuracy:

- 0:1: **0.8605**
- 0.5:1: **0.9212**
- 1:1: **0.9296**

According to the predefined selection rule, the **1:1 ratio was selected and locked**.

### Data efficiency

The 0.5:1 ratio delivered a substantial part of the full augmentation benefit while using only 576 synthetic observations. Mean macro-F1 increased from 0.8041 to 0.8950, whereas the 1:1 ratio reached 0.9219.

Therefore, 0.5:1 recovered approximately **77% of the full macro-F1 improvement** obtained between real-only and 1:1 training. It also recovered approximately **88% of the full balanced-accuracy improvement**. This indicates that the first half of the synthetic dataset provides most of the improvement, although the remaining synthetic observations further increase average performance and stability.

### Seed-dependent effects

The advantage of 1:1 over 0.5:1 was not uniform across all seeds. For validation macro-F1, the 1:1 configuration performed better for three of five seeds and worse for two. Much of its mean advantage resulted from the strong improvement at seed 52.

Nevertheless, 1:1 produced:

- the highest mean macro-F1;
- the highest mean balanced accuracy;
- the highest mean bird F1;
- the highest mean ROC-AUC;
- lower macro-F1 variability than 0.5:1;
- a higher worst-case macro-F1.

The selected ratio should therefore be interpreted as the strongest average and robustness-oriented configuration, not as universally superior for every initialization.

### Test-set policy

No new test evaluation was performed in this notebook. Ratio selection was conducted using validation data only. The selected 1:1 configuration had already been evaluated across five seeds in Notebook 09, where it achieved:

- mean test macro-F1: **0.9209 ± 0.0536**;
- mean test balanced accuracy: **0.9268 ± 0.0405**;
- mean test bird F1: **0.8654 ± 0.0895**;
- mean test ROC-AUC: **0.9863 ± 0.0182**.

Avoiding test evaluation of the rejected 0.5:1 candidate prevents the test set from influencing ratio selection.

### Limitations

1. Only three ratios were evaluated.
2. The 0.5:1 subset was selected once and fixed across training seeds.
3. Synthetic-subset-selection variability was not measured.
4. The 0.5:1 ratio is approximately 0.50087 because balanced integer class counts were required.
5. The official split is segment-based rather than session-independent.
6. Only transformation-based synthetic data were considered.
7. Five training seeds provide limited uncertainty estimation.

### Conclusion

The 1:1 synthetic-to-real ratio is retained as the preferred configuration because it provides the highest mean validation macro-F1 and the strongest overall stability across seeds. The 0.5:1 ratio remains a data-efficient alternative, recovering most of the full augmentation benefit with half as many synthetic observations.

The next experiment should evaluate whether the selected 1:1 augmentation strategy generalizes to previously unseen radar sessions using a session-independent data split.